# Model A: Encoder-Only Transformer from Scratch (Full Kaggle Yelp Dataset)

**Architecture:** ~40.8M parameter Pre-LN Encoder-Only Transformer (`d_model=512`, `8` layers, `8` heads, `d_ff=2048`, `max_seq_len=256`, Head+Tail `192+64` truncation, sinusoidal PE, masked mean pooling) built strictly from manual PyTorch primitives.  
**Task:** 3-class sentiment classification (`negative` = 1–2 stars, `neutral` = 3 stars, `positive` = 4–5 stars) across the full ~6.99M-review Kaggle Yelp dataset.

### Colab Setup Checklist
   **runtime type**:  **A100 GPU** + **High-RAM**.

## 1. Environment Setup, Drive Mount & A100 Verification

In [ ]:
import csv
from dataclasses import asdict
import json
import math
import os
from pathlib import Path
import random
import shutil
import subprocess
import sys
import time


subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kagglehub",
        "orjson",
        "transformers",
        "scipy",
        "scikit-learn",
        "matplotlib",
        "tqdm",
    ]
)

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required (Runtime > Change runtime type > A100)."
assert torch.cuda.is_bf16_supported(), "bfloat16 support is required (A100/Ampere GPU recommended)."

# Enable TensorFloat-32 on Ampere+ tensor cores for any fp32 matrix ops.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(
    f"GPU: {gpu_name} ({gpu_vram_gb:.1f} GB VRAM) | "
    f"bf16={torch.cuda.is_bf16_supported()} | PyTorch={torch.__version__}"
)

WORK_DIR = Path("/content/work")
DRIVE_DIR = Path("/content/drive/MyDrive/yelp_model_a")
TOKENIZED_WORK_DIR = WORK_DIR / "tokenized"
TOKENIZED_DRIVE_DIR = DRIVE_DIR / "tokenized"
# v2: optimizer param groups changed (embeddings excluded from weight decay), so old
# checkpoints are not resume-compatible. Use fresh dirs to avoid loading them by accident.
CKPT_DIR = DRIVE_DIR / "checkpoints_v2"
ARTIFACTS_DIR = DRIVE_DIR / "artifacts_v2"

for directory in (WORK_DIR, TOKENIZED_WORK_DIR, TOKENIZED_DRIVE_DIR, CKPT_DIR, ARTIFACTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB (79.3 GB VRAM) | bf16=True | PyTorch=2.14.1+cu130


## 2. Core Source Modules

In [2]:
from dataclasses import dataclass

LABEL2ID: dict[str, int] = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

ID2LABEL: dict[int, str] = {
    0: "negative",
    1: "neutral",
    2: "positive",
}


@dataclass
class ModelAConfig:
    """Architecture hyperparameters for the scratch encoder-only Transformer."""

    vocab_size: int = 30522
    pad_token_id: int = 0
    d_model: int = 512
    num_layers: int = 8
    num_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 256
    dropout: float = 0.10
    num_classes: int = 3
    head_tokens: int = 192
    tail_tokens: int = 64

    def __post_init__(self) -> None:
        # Token IDs are serialized to uint16 memmap storage, so vocab_size must fit in uint16.
        if self.vocab_size <= 0 or self.vocab_size > 65535:
            raise ValueError(f"vocab_size must be in [1, 65535], got {self.vocab_size}")
        if not (0 <= self.pad_token_id < self.vocab_size):
            raise ValueError(
                f"pad_token_id ({self.pad_token_id}) must be in [0, {self.vocab_size})"
            )
        if self.d_model <= 0 or self.num_heads <= 0 or self.d_model % self.num_heads != 0:
            raise ValueError(
                f"d_model ({self.d_model}) must be positive and divisible by num_heads ({self.num_heads})"
            )
        if self.num_layers <= 0 or self.d_ff <= 0 or self.num_classes <= 0:
            raise ValueError("num_layers, d_ff, and num_classes must be positive")
        if self.head_tokens < 0 or self.tail_tokens < 0:
            raise ValueError("head_tokens and tail_tokens must be non-negative")
        if self.head_tokens + self.tail_tokens != self.max_seq_len:
            raise ValueError(
                f"head_tokens ({self.head_tokens}) + tail_tokens ({self.tail_tokens}) "
                f"must equal max_seq_len ({self.max_seq_len})"
            )
        if not (0.0 <= self.dropout < 1.0):
            raise ValueError(f"dropout must be in [0.0, 1.0), got {self.dropout}")


@dataclass
class TrainingConfig:
    """Dataset, optimization, bucketing, and checkpointing hyperparameters."""

    seed: int = 42
    micro_batch_size: int = 256
    grad_accum_steps: int = 2
    epochs: int = 3
    peak_lr: float = 5e-4
    min_lr_ratio: float = 0.05
    warmup_ratio: float = 0.04
    weight_decay: float = 0.1
    label_smoothing: float = 0.1
    betas: tuple[float, float] = (0.9, 0.98)
    eps: float = 1e-8
    grad_clip_norm: float = 1.0
    bucket_multiple: int = 64
    mega_batch_mult: int = 50
    val_size: int = 20_000
    test_size: int = 40_000
    checkpoint_every_steps: int = 2000
    val_every_steps: int = 2000
    tokenizer_name: str = "bert-base-uncased"
    dataset_slug: str = "yelp-dataset/yelp-dataset"
    dataset_file: str = "yelp_academic_dataset_review.json"
    dataset_url: str = "https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset"
    dataset_license: str = "Yelp Dataset Agreement (non-commercial academic/educational use)"


<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 920 760" width="920" height="760">
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#475569"/>
    </marker>
    <marker id="arrow-blue" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#2563eb"/>
    </marker>
  </defs>
  <rect x="0" y="0" width="920" height="760" rx="14" fill="#f8fafc" stroke="#cbd5e1" stroke-width="1.5"/>
  <text x="30" y="36" font-family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif" font-size="18" font-weight="700" fill="#0f172a">Model A: ScratchTransformerClassifier Architecture</text>
  <text x="30" y="55" font-family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif" font-size="12" fill="#475569">Pre-LN Encoder-Only Transformer (~40.8M params) · d_model=512 · 8 layers · 8 heads · d_ff=2048 · max_seq_len=256</text>
  <rect x="40" y="80" width="330" height="52" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/>
  <text x="205" y="102" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">Inputs: input_ids &amp; attention_mask</text>
  <text x="205" y="120" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">mask = (input_ids != 0) · Shape: (B, T &lt;= 256)</text>
  <line x1="205" y1="132" x2="205" y2="158" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="40" y="160" width="330" height="58" rx="8" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="205" y="182" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">Token Embedding + sqrt(d_model) Scaling</text>
  <text x="205" y="200" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">nn.Embedding(30522, 512, pad=0) * sqrt(512)</text>
  <text x="380" y="193" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">(B, T, 512)</text>
  <line x1="205" y1="218" x2="205" y2="244" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="40" y="246" width="330" height="58" rx="8" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="205" y="268" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">SinusoidalPositionalEncoding</text>
  <text x="205" y="286" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">x + pe[:, :T].to(x.dtype) -&gt; Dropout(0.1)</text>
  <text x="380" y="279" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">(B, T, 512)</text>
  <line x1="205" y1="304" x2="205" y2="330" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="40" y="332" width="330" height="90" rx="10" fill="#ede9fe" stroke="#7c3aed" stroke-width="2"/>
  <rect x="52" y="342" width="66" height="22" rx="11" fill="#7c3aed"/>
  <text x="85" y="357" text-anchor="middle" font-family="system-ui, sans-serif" font-size="11" font-weight="700" fill="#ffffff">8 x Stack</text>
  <text x="225" y="358" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">PreLNEncoderBlock</text>
  <text x="205" y="382" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">Pre-LN Self-Attention + Residual</text>
  <text x="205" y="400" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">Pre-LN Position-Wise FFN + Residual</text>
  <text x="380" y="381" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">(B, T, 512)</text>
  <path d="M 370 355 L 470 110" stroke="#7c3aed" stroke-width="1.5" stroke-dasharray="5,4" fill="none"/>
  <path d="M 370 405 L 470 590" stroke="#7c3aed" stroke-width="1.5" stroke-dasharray="5,4" fill="none"/>
  <line x1="205" y1="422" x2="205" y2="448" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="40" y="450" width="330" height="46" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
  <text x="205" y="471" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">Final LayerNorm</text>
  <text x="205" y="487" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">nn.LayerNorm(512)</text>
  <line x1="205" y1="496" x2="205" y2="522" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="40" y="524" width="330" height="62" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="205" y="546" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">Masked Mean Pooling</text>
  <text x="205" y="564" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">sum(x * mask) / clamp(sum(mask), min=1.0)</text>
  <text x="205" y="579" text-anchor="middle" font-family="system-ui, sans-serif" font-size="11" fill="#475569">Averages only non-padding token states</text>
  <text x="380" y="559" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">(B, 512)</text>
  <line x1="205" y1="586" x2="205" y2="612" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="40" y="614" width="330" height="58" rx="8" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/>
  <text x="205" y="636" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">Classification Head</text>
  <text x="205" y="654" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">Dropout(0.1) -&gt; nn.Linear(512, 3)</text>
  <text x="380" y="647" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">(B, 3)</text>
  <line x1="205" y1="672" x2="205" y2="696" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <rect x="75" y="698" width="260" height="40" rx="20" fill="#0f172a"/>
  <text x="205" y="723" text-anchor="middle" font-family="system-ui, sans-serif" font-size="11" font-weight="700" fill="#ffffff">Logits: [negative, neutral, positive]</text>
  <rect x="470" y="80" width="420" height="520" rx="12" fill="#ffffff" stroke="#7c3aed" stroke-width="2"/>
  <text x="680" y="106" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#5b21b6">Inside 1 x PreLNEncoderBlock</text>
  <text x="680" y="130" text-anchor="middle" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">Input x (B, T, 512)</text>
  <line x1="680" y1="138" x2="680" y2="162" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <circle cx="680" cy="148" r="3.5" fill="#2563eb"/>
  <path d="M 680 148 L 510 148 L 510 330 L 664 330" stroke="#2563eb" stroke-width="1.8" fill="none" marker-end="url(#arrow-blue)"/>
  <text x="502" y="245" font-family="system-ui, sans-serif" font-size="11" fill="#2563eb" transform="rotate(-90 502 245)">Residual Skip</text>
  <rect x="550" y="164" width="260" height="34" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.3"/>
  <text x="680" y="185" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">ln1: LayerNorm(512)</text>
  <line x1="680" y1="198" x2="680" y2="216" stroke="#475569" stroke-width="1.6" marker-end="url(#arrow)"/>
  <rect x="530" y="218" width="300" height="84" rx="8" fill="#e0f2fe" stroke="#0284c7" stroke-width="1.5"/>
  <text x="680" y="238" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">ManualMultiHeadSelfAttention</text>
  <text x="680" y="255" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">Q, K, V -&gt; (B, 8, T, 64) · scores / sqrt(64)</text>
  <text x="680" y="271" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">fp32 masked softmax (-inf on pads)</text>
  <text x="680" y="287" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">attn_drop(0.1) -&gt; @ V -&gt; w_o -&gt; drop1(0.1)</text>
  <line x1="680" y1="302" x2="680" y2="316" stroke="#475569" stroke-width="1.6" marker-end="url(#arrow)"/>
  <circle cx="680" cy="330" r="13" fill="#eff6ff" stroke="#2563eb" stroke-width="1.8"/>
  <text x="680" y="335" text-anchor="middle" font-family="system-ui, sans-serif" font-size="14" font-weight="700" fill="#2563eb">+</text>
  <line x1="680" y1="343" x2="680" y2="378" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <circle cx="680" cy="362" r="3.5" fill="#2563eb"/>
  <path d="M 680 362 L 510 362 L 510 536 L 664 536" stroke="#2563eb" stroke-width="1.8" fill="none" marker-end="url(#arrow-blue)"/>
  <text x="502" y="455" font-family="system-ui, sans-serif" font-size="11" fill="#2563eb" transform="rotate(-90 502 455)">Residual Skip</text>
  <rect x="550" y="380" width="260" height="34" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.3"/>
  <text x="680" y="401" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">ln2: LayerNorm(512)</text>
  <line x1="680" y1="414" x2="680" y2="432" stroke="#475569" stroke-width="1.6" marker-end="url(#arrow)"/>
  <rect x="530" y="434" width="300" height="72" rx="8" fill="#f3e8ff" stroke="#9333ea" stroke-width="1.5"/>
  <text x="680" y="454" text-anchor="middle" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">PositionWiseFeedForward</text>
  <text x="680" y="472" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">fc1: 512 -&gt; 2048 · GELU · drop(0.1)</text>
  <text x="680" y="489" text-anchor="middle" font-family="monospace" font-size="11" fill="#334155">fc2: 2048 -&gt; 512 · drop2(0.1)</text>
  <line x1="680" y1="506" x2="680" y2="522" stroke="#475569" stroke-width="1.6" marker-end="url(#arrow)"/>
  <circle cx="680" cy="536" r="13" fill="#eff6ff" stroke="#2563eb" stroke-width="1.8"/>
  <text x="680" y="541" text-anchor="middle" font-family="system-ui, sans-serif" font-size="14" font-weight="700" fill="#2563eb">+</text>
  <line x1="680" y1="549" x2="680" y2="572" stroke="#475569" stroke-width="1.8" marker-end="url(#arrow)"/>
  <text x="680" y="588" text-anchor="middle" font-family="monospace" font-size="11" font-weight="600" fill="#2563eb">Output x (B, T, 512)</text>
  <rect x="470" y="618" width="420" height="120" rx="10" fill="#f1f5f9" stroke="#cbd5e1" stroke-width="1.5"/>
  <text x="488" y="641" font-family="system-ui, sans-serif" font-size="13" font-weight="700" fill="#0f172a">Key Implementation Details (src/model.py)</text>
  <text x="488" y="662" font-family="system-ui, sans-serif" font-size="11" fill="#475569">- Weight Init: Linear/Emb ~ N(0, 0.02); pad_idx=0 zeroed.</text>
  <text x="488" y="681" font-family="system-ui, sans-serif" font-size="11" fill="#475569">- Residual Scaling: w_o &amp; fc2 std = 0.02 / sqrt(2 * 8) = 0.005.</text>
  <text x="488" y="700" font-family="system-ui, sans-serif" font-size="11" fill="#475569">- AMP Stability: Attention scores cast to fp32 before -inf softmax.</text>
  <text x="488" y="719" font-family="system-ui, sans-serif" font-size="11" fill="#475569">- AdamW Groups: Weight decay on 2D+ tensors only (not biases/LN/embeddings).</text>
</svg>

In [3]:
"""PyTorch encoder-only Transformer classifier built strictly from manual primitives."""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from src.config import ModelAConfig
except ImportError:
    pass


class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding added to scaled token embeddings."""

    def __init__(self, d_model: int, max_len: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: d_model // 2])
        # Register as a non-trainable buffer with leading batch dim (1, max_len, d_model).
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Match input dtype so bf16/fp16 autocast activations are not silently promoted to fp32.
        x = x + self.pe[:, : x.size(1)].to(dtype=x.dtype)
        return self.dropout(x)


class ManualMultiHeadSelfAttention(nn.Module):
    """Manual multi-head self-attention with fp32 softmax masking for AMP stability."""

    def __init__(self, d_model: int, num_heads: int, dropout: float) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model, bias=True)
        self.w_k = nn.Linear(d_model, d_model, bias=True)
        self.w_v = nn.Linear(d_model, d_model, bias=True)
        self.w_o = nn.Linear(d_model, d_model, bias=True)
        self.attn_drop = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        bsz, seq_len, _ = x.shape

        # Project and reshape from (B, T, D) into (B, H, T, d_k).
        q = self.w_q(x).view(bsz, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(bsz, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(bsz, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Compute masked softmax in fp32 to prevent overflow/underflow under bf16/fp16 autocast.
        scores = scores.to(torch.float32)
        if key_padding_mask is not None:
            pad_positions = ~key_padding_mask.bool()
            scores = scores.masked_fill(pad_positions[:, None, None, :], float("-inf"))

        attn_probs = F.softmax(scores, dim=-1).to(v.dtype)
        attn_probs = self.attn_drop(attn_probs)

        out = torch.matmul(attn_probs, v)
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.d_model)
        return self.w_o(out)


class PositionWiseFeedForward(nn.Module):
    """Two-layer position-wise FFN with GELU activation."""

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.drop(self.act(self.fc1(x))))


class PreLNEncoderBlock(nn.Module):
    """Pre-LayerNorm encoder block with residual self-attention and FFN sublayers."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = ManualMultiHeadSelfAttention(d_model, num_heads, dropout)
        self.drop1 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        x = x + self.drop1(self.attn(self.ln1(x), key_padding_mask))
        x = x + self.drop2(self.ffn(self.ln2(x)))
        return x


class ScratchTransformerClassifier(nn.Module):
    """Encoder-only Transformer classifier with masked mean pooling over token states."""

    def __init__(self, config: ModelAConfig) -> None:
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        self.pad_token_id = config.pad_token_id

        self.token_emb = nn.Embedding(
            config.vocab_size,
            config.d_model,
            padding_idx=config.pad_token_id,
        )
        self.pos_enc = SinusoidalPositionalEncoding(
            config.d_model,
            config.max_seq_len,
            config.dropout,
        )
        self.layers = nn.ModuleList(
            [
                PreLNEncoderBlock(
                    config.d_model,
                    config.num_heads,
                    config.d_ff,
                    config.dropout,
                )
                for _ in range(config.num_layers)
            ]
        )
        self.final_ln = nn.LayerNorm(config.d_model)
        self.head_drop = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.d_model, config.num_classes)

        self._init_weights()

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.padding_idx is not None:
                    with torch.no_grad():
                        module.weight[module.padding_idx].zero_()
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        # Scale residual output projections by 1 / sqrt(2 * num_layers) to stabilize
        # variance growth across stacked Pre-LN residual additions.
        resid_std = 0.02 / math.sqrt(2 * self.config.num_layers)
        for layer in self.layers:
            nn.init.normal_(layer.attn.w_o.weight, mean=0.0, std=resid_std)
            nn.init.normal_(layer.ffn.fc2.weight, mean=0.0, std=resid_std)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        mask = (input_ids != self.pad_token_id) if attention_mask is None else attention_mask.bool()

        # Scale embeddings by sqrt(d_model) so token magnitude balances unit sinusoidal PE.
        x = self.token_emb(input_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.final_ln(x)

        # Masked mean pooling averages only non-pad positions and clamps denom >= 1.0 for safety.
        mask_f = mask.unsqueeze(-1).type_as(x)
        pooled = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
        return self.classifier(self.head_drop(pooled))

    def count_parameters(self) -> dict[str, int]:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        embedding = sum(p.numel() for p in self.token_emb.parameters())
        return {
            "total": total,
            "trainable": trainable,
            "embedding": embedding,
            "non_embedding": total - embedding,
        }


def build_adamw_param_groups(model: nn.Module, weight_decay: float) -> list[dict]:
    """Decay 2D+ weight matrices; exempt biases, LayerNorm gains, and token embeddings."""
    decay_params: list[torch.nn.Parameter] = []
    no_decay_params: list[torch.nn.Parameter] = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim >= 2 and "token_emb" not in name:
            decay_params.append(param)
        else:
            no_decay_params.append(param)

    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]


In [4]:
# === AUTO-GENERATED FROM src/data.py ===
"""Streaming JSONL preprocessing, CSR uint16 memmap dataset, and bucketed collation."""

from array import array
import hashlib
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import torch
from torch.utils.data import Dataset

try:
    import orjson

    def _json_loads(raw_line: bytes) -> Any:
        return orjson.loads(raw_line)

except ImportError:

    def _json_loads(raw_line: bytes) -> Any:
        return json.loads(raw_line)


try:
    from src.config import ID2LABEL, ModelAConfig, TrainingConfig
except ImportError:
    pass


def map_stars_to_label(stars: Any) -> int | None:
    """Map Yelp star rating (1..5) to 3-class ID: {1,2}->0, {3}->1, {4,5}->2."""
    if stars is None or isinstance(stars, bool):
        return None
    try:
        val = float(stars)
    except (TypeError, ValueError):
        return None
    if math.isnan(val) or math.isinf(val):
        return None

    rounded = int(round(val))
    if rounded in (1, 2):
        return 0
    if rounded == 3:
        return 1
    if rounded in (4, 5):
        return 2
    return None


def normalize_for_dedup(text: str) -> str:
    """Lowercase and collapse whitespace for exact-duplicate detection."""
    return " ".join(text.strip().lower().split())


def head_tail_truncate(
    token_ids: list[int],
    max_len: int = 256,
    head_tokens: int = 192,
    tail_tokens: int = 64,
) -> list[int]:
    """Keep leading head_tokens and trailing tail_tokens (preserving [CLS] and [SEP])."""
    if len(token_ids) <= max_len:
        return list(token_ids)
    head = token_ids[:head_tokens] if head_tokens > 0 else []
    # Guard tail_tokens == 0 because Python slices [-0:] as [0:].
    tail = token_ids[-tail_tokens:] if tail_tokens > 0 else []
    return (head + tail)[:max_len]


def _count_classes(labels_subset: np.ndarray) -> dict[str, int]:
    return {
        ID2LABEL[cid]: int(np.sum(labels_subset == cid))
        for cid in sorted(ID2LABEL.keys())
    }


def build_memmap_dataset(
    json_path: str | Path,
    out_dir: str | Path,
    tokenizer: Any,
    model_cfg: ModelAConfig,
    train_cfg: TrainingConfig,
) -> dict[str, Any]:
    """Stream Yelp JSONL into deduplicated CSR uint16 memmap files and stratified splits."""
    assert tokenizer.vocab_size <= 65535, (
        f"Tokenizer vocab_size ({tokenizer.vocab_size}) exceeds uint16 max (65535)"
    )

    src_path = Path(json_path)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    tokens_bin_path = out_path / "tokens.bin"
    chunk_size = 16_384

    raw_count = 0
    dropped_invalid = 0
    dropped_duplicate = 0
    kept_count = 0
    total_tokens = 0

    seen_hashes: set[bytes] = set()
    src_offsets_buf = array("q")  # byte offset of each kept record in the source JSONL
    offsets_buf = array("q")
    lengths_buf = array("H")
    labels_buf = array("B")

    chunk_texts: list[str] = []
    chunk_labels: list[int] = []

    def _flush_chunk(tokens_file: Any) -> None:
        nonlocal kept_count, total_tokens
        if not chunk_texts:
            return
        encoded = tokenizer(
            chunk_texts,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        for ids, lbl in zip(encoded["input_ids"], chunk_labels):
            trunc = head_tail_truncate(
                ids,
                max_len=model_cfg.max_seq_len,
                head_tokens=model_cfg.head_tokens,
                tail_tokens=model_cfg.tail_tokens,
            )
            seq_len = len(trunc)
            offsets_buf.append(total_tokens)
            lengths_buf.append(seq_len)
            labels_buf.append(lbl)
            if seq_len > 0:
                tokens_file.write(np.asarray(trunc, dtype=np.uint16).tobytes())
            total_tokens += seq_len
            kept_count += 1
        chunk_texts.clear()
        chunk_labels.clear()

    with src_path.open("rb") as f_in, tokens_bin_path.open("wb") as f_tok:
        line_start = 0
        for raw_line in f_in:
            line_pos = line_start
            line_start += len(raw_line)
            stripped = raw_line.strip()
            if not stripped:
                continue
            raw_count += 1
            try:
                record = _json_loads(stripped)
            except (ValueError, TypeError, UnicodeDecodeError):
                dropped_invalid += 1
                continue

            if not isinstance(record, dict):
                dropped_invalid += 1
                continue

            label = map_stars_to_label(record.get("stars"))
            text = record.get("text")
            if label is None or not isinstance(text, str):
                dropped_invalid += 1
                continue

            norm_text = normalize_for_dedup(text)
            if not norm_text:
                dropped_invalid += 1
                continue

            # 64-bit keyed-free blake2b digest keeps the ~7M-entry dedup set compact in RAM.
            digest = hashlib.blake2b(norm_text.encode("utf-8"), digest_size=8).digest()
            if digest in seen_hashes:
                dropped_duplicate += 1
                continue
            seen_hashes.add(digest)

            src_offsets_buf.append(line_pos)
            chunk_texts.append(text)
            chunk_labels.append(label)
            if len(chunk_texts) >= chunk_size:
                _flush_chunk(f_tok)

        _flush_chunk(f_tok)

    if kept_count == 0:
        raise ValueError(f"No valid records found in {src_path}")
    if train_cfg.val_size + train_cfg.test_size >= kept_count:
        raise ValueError(
            f"val_size ({train_cfg.val_size}) + test_size ({train_cfg.test_size}) "
            f"must be strictly less than kept_count ({kept_count})"
        )

    offsets_arr = np.asarray(offsets_buf, dtype=np.int64)
    lengths_arr = np.asarray(lengths_buf, dtype=np.uint16)
    labels_arr = np.asarray(labels_buf, dtype=np.uint8)

    np.save(out_path / "offsets.npy", offsets_arr)
    np.save(out_path / "lengths.npy", lengths_arr)
    np.save(out_path / "labels.npy", labels_arr)

    all_indices = np.arange(kept_count, dtype=np.int32)
    sss_test = StratifiedShuffleSplit(
        n_splits=1,
        test_size=train_cfg.test_size,
        random_state=train_cfg.seed,
    )
    trainval_pos, test_pos = next(sss_test.split(all_indices, labels_arr))
    trainval_idx = all_indices[trainval_pos].astype(np.int32)
    test_idx = all_indices[test_pos].astype(np.int32)

    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=train_cfg.val_size,
        random_state=train_cfg.seed,
    )
    train_rel, val_rel = next(sss_val.split(trainval_idx, labels_arr[trainval_idx]))
    train_idx = trainval_idx[train_rel].astype(np.int32)
    val_idx = trainval_idx[val_rel].astype(np.int32)

    np.save(out_path / "train_idx.npy", train_idx)
    np.save(out_path / "val_idx.npy", val_idx)
    np.save(out_path / "test_idx.npy", test_idx)

    # Export original (untruncated, cased) test review texts in test_idx order so error
    # analysis shows real text instead of lowercased, head+tail-truncated decoded tokens.
    src_offsets = np.asarray(src_offsets_buf, dtype=np.int64)
    test_texts: list[str] = [""] * len(test_idx)
    read_order = np.argsort(src_offsets[test_idx], kind="mergesort")  # sequential disk reads
    with src_path.open("rb") as f_src:
        for j in read_order:
            f_src.seek(int(src_offsets[test_idx[j]]))
            test_texts[int(j)] = _json_loads(f_src.readline().strip())["text"]
    with (out_path / "test_texts.jsonl").open("w", encoding="utf-8") as f_txt:
        for t in test_texts:
            f_txt.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")

    manifest: dict[str, Any] = {
        "dataset_slug": train_cfg.dataset_slug,
        "dataset_file": train_cfg.dataset_file,
        "dataset_url": train_cfg.dataset_url,
        "dataset_license": train_cfg.dataset_license,
        "citation": (
            "Yelp Open Dataset (https://www.yelp.com/dataset), "
            f"mirror: {train_cfg.dataset_url}"
        ),
        "tokenizer_name": train_cfg.tokenizer_name,
        "seed": train_cfg.seed,
        "max_seq_len": model_cfg.max_seq_len,
        "head_tokens": model_cfg.head_tokens,
        "tail_tokens": model_cfg.tail_tokens,
        "counts": {
            "raw": raw_count,
            "dropped_invalid": dropped_invalid,
            "dropped_duplicate": dropped_duplicate,
            "deduped": kept_count,
            "total_tokens": total_tokens,
        },
        "splits": {
            "train": {
                "size": int(len(train_idx)),
                "class_counts": _count_classes(labels_arr[train_idx]),
                "sha256": hashlib.sha256(train_idx.tobytes()).hexdigest(),
            },
            "val": {
                "size": int(len(val_idx)),
                "class_counts": _count_classes(labels_arr[val_idx]),
                "sha256": hashlib.sha256(val_idx.tobytes()).hexdigest(),
            },
            "test": {
                "size": int(len(test_idx)),
                "class_counts": _count_classes(labels_arr[test_idx]),
                "sha256": hashlib.sha256(test_idx.tobytes()).hexdigest(),
            },
        },
    }

    with (out_path / "split_manifest.json").open("w", encoding="utf-8") as f_manifest:
        json.dump(manifest, f_manifest, indent=2)

    return manifest


class YelpMemmapDataset(Dataset):
    """CSR memmap dataset that opens read-only file mappings lazily per worker."""

    def __init__(self, data_dir: str | Path, indices: np.ndarray) -> None:
        base = Path(data_dir)
        self.tokens_path = str(base / "tokens.bin")
        self.offsets_path = str(base / "offsets.npy")
        self.lengths_path = str(base / "lengths.npy")
        self.labels_path = str(base / "labels.npy")
        self.indices = np.asarray(indices, dtype=np.int64)

        # Defer mmap creation until __getitem__ so DataLoader fork/spawn workers
        # never share open mmap descriptors across processes.
        self._tokens: np.memmap | None = None
        self._offsets: np.ndarray | None = None
        self._lengths: np.ndarray | None = None
        self._labels: np.ndarray | None = None

    def _ensure_open(self) -> None:
        if self._tokens is None:
            self._tokens = np.memmap(self.tokens_path, dtype=np.uint16, mode="r")
            self._offsets = np.load(self.offsets_path, mmap_mode="r")
            self._lengths = np.load(self.lengths_path, mmap_mode="r")
            self._labels = np.load(self.labels_path, mmap_mode="r")

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, item: int) -> tuple[np.ndarray, int]:
        self._ensure_open()
        assert self._tokens is not None
        assert self._offsets is not None
        assert self._lengths is not None
        assert self._labels is not None

        idx = int(self.indices[item])
        s = int(self._offsets[idx])
        e = s + int(self._lengths[idx])
        # Copy slice into a writable int64 array to avoid read-only memmap warnings.
        tokens = np.array(self._tokens[s:e], dtype=np.int64)
        return tokens, int(self._labels[idx])


class BucketCollateFn:
    """Pad variable-length sequences to multiples of bucket_multiple for torch.compile."""

    def __init__(
        self,
        pad_token_id: int = 0,
        bucket_multiple: int = 64,
        max_seq_len: int = 256,
    ) -> None:
        if bucket_multiple <= 0 or max_seq_len <= 0:
            raise ValueError("bucket_multiple and max_seq_len must be positive")
        self.pad_token_id = pad_token_id
        self.bucket_multiple = bucket_multiple
        self.max_seq_len = max_seq_len

    def __call__(
        self,
        batch: list[tuple[np.ndarray, int]],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if not batch:
            return (
                torch.empty((0, 0), dtype=torch.long),
                torch.empty((0, 0), dtype=torch.bool),
                torch.empty((0,), dtype=torch.long),
            )

        seqs, labels = zip(*batch)
        max_len_in_batch = max(len(s) for s in seqs)

        # Snap padded length to a fixed set of multiples of bucket_multiple (e.g. 64, 128, 192, 256)
        # so torch.compile sees at most max_seq_len // bucket_multiple distinct input shapes.
        bucketed = (
            (max_len_in_batch + self.bucket_multiple - 1) // self.bucket_multiple
        ) * self.bucket_multiple
        padded_len = min(self.max_seq_len, max(self.bucket_multiple, bucketed))

        bsz = len(seqs)
        input_ids = torch.full((bsz, padded_len), self.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((bsz, padded_len), dtype=torch.bool)
        labels_tensor = torch.tensor(labels, dtype=torch.long)

        for i, seq in enumerate(seqs):
            seq_len = min(len(seq), padded_len)
            if seq_len > 0:
                input_ids[i, :seq_len] = torch.as_tensor(seq[:seq_len], dtype=torch.long)
                attention_mask[i, :seq_len] = True

        return input_ids, attention_mask, labels_tensor


In [5]:
# === AUTO-GENERATED FROM src/sampler.py ===
"""Mega-batch length-grouped epoch scheduling and O(1) resumable batch sampler."""

from collections.abc import Iterator

import numpy as np
from torch.utils.data import Sampler


def build_epoch_batches(
    indices: np.ndarray,
    lengths: np.ndarray,
    batch_size: int,
    seed: int,
    epoch: int,
    mega_batch_mult: int = 50,
    drop_last: bool = True,
) -> list[np.ndarray]:
    """Build deterministic length-grouped micro-batches for a given epoch."""
    if batch_size <= 0:
        raise ValueError(f"batch_size must be positive, got {batch_size}")
    if mega_batch_mult <= 0:
        raise ValueError(f"mega_batch_mult must be positive, got {mega_batch_mult}")

    # Offset seed by a large prime per epoch so every epoch draws an independent schedule
    # while remaining 100% reproducible on checkpoint resume.
    rng = np.random.default_rng(seed + epoch * 1_000_003)
    perm = rng.permutation(np.asarray(indices))

    mega_size = batch_size * mega_batch_mult
    batches: list[np.ndarray] = []

    for mega_start in range(0, len(perm), mega_size):
        mb = perm[mega_start : mega_start + mega_size]
        # Stable-sort within each mega-batch by sequence length so each micro-batch groups
        # similarly sized samples, minimizing bucket padding without hurting batch diversity.
        order = np.argsort(lengths[mb], kind="mergesort")
        mb_sorted = mb[order]
        for b_start in range(0, len(mb_sorted), batch_size):
            batch = mb_sorted[b_start : b_start + batch_size]
            if drop_last and len(batch) < batch_size:
                continue
            batches.append(batch)

    if batches:
        # Permute indices rather than the list itself so ragged tail batches (drop_last=False)
        # never trigger NumPy inhomogeneous array errors.
        batch_order = rng.permutation(len(batches))
        batches = [batches[i] for i in batch_order]

    return batches


class ResumableBatchSampler(Sampler):
    """O(1)-resumable batch sampler that slices precomputed epoch batches at start_batch_idx."""

    def __init__(self, batches: list[np.ndarray], start_batch_idx: int = 0) -> None:
        super().__init__()
        if start_batch_idx < 0:
            raise ValueError(f"start_batch_idx must be non-negative, got {start_batch_idx}")
        self.batches = batches
        self.start_batch_idx = start_batch_idx

    def __iter__(self) -> Iterator[np.ndarray]:
        yield from self.batches[self.start_batch_idx :]

    def __len__(self) -> int:
        return max(0, len(self.batches) - self.start_batch_idx)


In [6]:
# === AUTO-GENERATED FROM src/metrics.py ===
"""Evaluation metrics, 2D logit bias tuning, GPU latency benchmarking, and I/O helpers."""

import csv
import os
from pathlib import Path
import shutil
import time
from typing import Any

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
import torch

try:
    from src.config import ID2LABEL
except ImportError:
    pass


def compute_classification_metrics(
    y_true: np.ndarray | list[int],
    y_pred: np.ndarray | list[int],
) -> dict[str, Any]:
    """Compute overall accuracy, macro/weighted F1, per-class metrics, and 3x3 CM."""
    y_true_arr = np.asarray(y_true, dtype=np.int64)
    y_pred_arr = np.asarray(y_pred, dtype=np.int64)
    class_ids = sorted(ID2LABEL.keys())
    label_names = [ID2LABEL[cid] for cid in class_ids]

    acc = float(accuracy_score(y_true_arr, y_pred_arr))
    macro_f1 = float(
        f1_score(y_true_arr, y_pred_arr, labels=class_ids, average="macro", zero_division=0)
    )
    weighted_f1 = float(
        f1_score(y_true_arr, y_pred_arr, labels=class_ids, average="weighted", zero_division=0)
    )

    prec, rec, f1, supp = precision_recall_fscore_support(
        y_true_arr,
        y_pred_arr,
        labels=class_ids,
        zero_division=0,
    )
    per_class = {
        ID2LABEL[cid]: {
            "precision": float(prec[i]),
            "recall": float(rec[i]),
            "f1": float(f1[i]),
            "support": int(supp[i]),
        }
        for i, cid in enumerate(class_ids)
    }

    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=class_ids)
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "per_class": per_class,
        "confusion_matrix": {
            "labels": label_names,
            "matrix": cm.tolist(),
        },
    }


def tune_logit_biases(
    val_logits: np.ndarray,
    val_labels: np.ndarray,
) -> tuple[np.ndarray, float]:
    """Optimize 2D additive logit biases [b0, b1, 0.0] to maximize validation Macro-F1.

    Because argmax(logits + b) is invariant to adding a constant across all classes,
    fixing b2 = 0.0 removes the redundant degree of freedom. Multi-start grid search
    plus Nelder-Mead handles the piecewise-constant step surface where gradients vanish.
    """
    logits = np.asarray(val_logits, dtype=np.float64)
    labels = np.asarray(val_labels, dtype=np.int64)
    class_ids = [0, 1, 2]

    def _score_bias(b0: float, b1: float) -> float:
        bias = np.array([b0, b1, 0.0], dtype=np.float64)
        preds = (logits + bias).argmax(axis=-1)
        return float(
            f1_score(labels, preds, labels=class_ids, average="macro", zero_division=0)
        )

    # Always anchor on zero bias so tuned F1 can never regress below baseline.
    best_b = (0.0, 0.0)
    best_f1 = _score_bias(0.0, 0.0)

    candidates: list[tuple[float, float, float]] = [(best_f1, 0.0, 0.0)]

    # Include inverse-frequency log-prior bias relative to class 2 as an extra start.
    counts = np.bincount(labels, minlength=3).astype(np.float64)
    if counts[2] > 0 and counts[0] > 0 and counts[1] > 0:
        prior_b0 = float(np.log(counts[2] / counts[0]))
        prior_b1 = float(np.log(counts[2] / counts[1]))
        candidates.append((_score_bias(prior_b0, prior_b1), prior_b0, prior_b1))

    grid = np.linspace(-2.0, 2.0, 13)
    for b0 in grid:
        for b1 in grid:
            f1 = _score_bias(float(b0), float(b1))
            candidates.append((f1, float(b0), float(b1)))
            if f1 > best_f1:
                best_f1 = f1
                best_b = (float(b0), float(b1))

    # Polish the top distinct starts with derivative-free Nelder-Mead simplex search.
    candidates.sort(key=lambda item: item[0], reverse=True)
    starts: list[tuple[float, float]] = [(0.0, 0.0)]
    for _, b0, b1 in candidates:
        if (b0, b1) not in starts:
            starts.append((b0, b1))
        if len(starts) >= 5:
            break

    for s0, s1 in starts:
        res = minimize(
            lambda b: -_score_bias(float(b[0]), float(b[1])),
            x0=np.array([s0, s1], dtype=np.float64),
            method="Nelder-Mead",
            options={"xatol": 1e-3, "fatol": 1e-5, "maxiter": 150},
        )
        opt_b0, opt_b1 = float(res.x[0]), float(res.x[1])
        opt_f1 = _score_bias(opt_b0, opt_b1)
        if opt_f1 > best_f1:
            best_f1 = opt_f1
            best_b = (opt_b0, opt_b1)

    return np.array([best_b[0], best_b[1], 0.0], dtype=np.float64), float(best_f1)


def _stage_batches(
    sample_batches: list[Any],
    device: torch.device,
) -> list[tuple[torch.Tensor, torch.Tensor | None]]:
    if not sample_batches:
        raise ValueError("sample_batches must be non-empty")
    staged: list[tuple[torch.Tensor, torch.Tensor | None]] = []
    for batch in sample_batches:
        if isinstance(batch, dict):
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch.get("attention_mask")
            mask = mask.to(device, non_blocking=True) if mask is not None else None
        else:
            ids = batch[0].to(device, non_blocking=True)
            mask = (
                batch[1].to(device, non_blocking=True)
                if len(batch) > 1 and batch[1] is not None
                else None
            )
        staged.append((ids, mask))
    return staged


def _bench_single_config(
    model: torch.nn.Module,
    staged_batches: list[tuple[torch.Tensor, torch.Tensor | None]],
    device: torch.device,
    warmup: int,
    steps: int,
) -> dict[str, float]:
    if device.type == "cuda":
        torch.cuda.synchronize(device)

    for i in range(warmup):
        ids, mask = staged_batches[i % len(staged_batches)]
        _ = model(ids, mask)

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    latencies_ms: list[float] = []
    total_samples = 0

    for i in range(steps):
        ids, mask = staged_batches[i % len(staged_batches)]
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        t0 = time.perf_counter()
        _ = model(ids, mask)
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        t1 = time.perf_counter()
        latencies_ms.append((t1 - t0) * 1000.0)
        total_samples += int(ids.size(0))

    arr = np.asarray(latencies_ms, dtype=np.float64)
    mean_ms_per_batch = float(np.mean(arr))
    total_sec = float(np.sum(arr)) / 1000.0
    mean_bs = total_samples / max(steps, 1)

    return {
        "mean_ms_per_batch": mean_ms_per_batch,
        "p50_ms_per_batch": float(np.percentile(arr, 50)),
        "p95_ms_per_batch": float(np.percentile(arr, 95)),
        "mean_ms_per_sample": mean_ms_per_batch / max(mean_bs, 1.0),
        "samples_per_sec": (total_samples / total_sec) if total_sec > 0.0 else 0.0,
    }


def benchmark_gpu_latency(
    model: torch.nn.Module,
    sample_batches_bs1: list[Any],
    sample_batches_bs32: list[Any],
    device: torch.device | str,
    warmup: int = 20,
    steps: int = 200,
) -> dict[str, dict[str, float]]:
    """Measure synchronized inference latency (ms/batch, p50, p95, throughput) for bs=1 and bs=32."""
    dev = torch.device(device)
    model.eval()

    staged_bs1 = _stage_batches(sample_batches_bs1, dev)
    staged_bs32 = _stage_batches(sample_batches_bs32, dev)

    with torch.inference_mode():
        bs1_stats = _bench_single_config(model, staged_bs1, dev, warmup=warmup, steps=steps)
        bs32_stats = _bench_single_config(model, staged_bs32, dev, warmup=warmup, steps=steps)

    return {"bs1": bs1_stats, "bs32": bs32_stats}


def strip_orig_mod(state_dict: dict[str, Any]) -> dict[str, Any]:
    """Remove torch.compile '_orig_mod.' key prefix when exporting portable checkpoints."""
    prefix = "_orig_mod."
    return {
        (k[len(prefix) :] if k.startswith(prefix) else k): v
        for k, v in state_dict.items()
    }


def atomic_drive_save(
    payload: Any,
    local_tmp_path: str | Path,
    drive_target_path: str | Path,
) -> Path:
    """Save checkpoint locally first, copy to <target>.tmp on Drive, then os.replace atomically.

    Writing torch.save streams directly onto a FUSE-mounted Google Drive risks truncated
    zip archives if Colab disconnects mid-write; atomic replacement prevents corruption.
    """
    local_tmp = Path(local_tmp_path)
    drive_target = Path(drive_target_path)
    local_tmp.parent.mkdir(parents=True, exist_ok=True)
    drive_target.parent.mkdir(parents=True, exist_ok=True)
    drive_tmp = drive_target.with_name(drive_target.name + ".tmp")

    try:
        torch.save(payload, local_tmp)
        shutil.copyfile(local_tmp, drive_tmp)
        os.replace(drive_tmp, drive_target)
    finally:
        if local_tmp.exists():
            local_tmp.unlink()
        if drive_tmp.exists():
            drive_tmp.unlink()

    return drive_target


def plot_training_curves(history: list[dict[str, Any]], out_path: str | Path) -> Path:
    """Plot train/val loss and validation Macro-F1 / accuracy curves to a PNG file."""
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    def _series(keys: tuple[str, ...]) -> tuple[list[float], list[float]]:
        xs: list[float] = []
        ys: list[float] = []
        for idx, row in enumerate(history):
            step = row.get("step", row.get("global_step", idx))
            for key in keys:
                if key in row and row[key] is not None:
                    xs.append(float(step))
                    ys.append(float(row[key]))
                    break
        return xs, ys

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    ax_loss = axes[0]
    tx, ty = _series(("train_loss", "loss"))
    vx, vy = _series(("val_loss",))
    if tx:
        ax_loss.plot(tx, ty, label="Train Loss", linewidth=1.8)
    if vx:
        ax_loss.plot(vx, vy, label="Val Loss", marker="o", markersize=4, linewidth=1.8)
    ax_loss.set_title("Training & Validation Loss")
    ax_loss.set_xlabel("Optimizer Step")
    ax_loss.set_ylabel("Cross-Entropy Loss")
    ax_loss.grid(True, alpha=0.3)
    if tx or vx:
        ax_loss.legend()

    ax_metric = axes[1]
    fx, fy = _series(("val_macro_f1", "macro_f1"))
    ax_x, ax_y = _series(("val_accuracy", "accuracy"))
    if fx:
        ax_metric.plot(fx, fy, label="Val Macro-F1", marker="o", markersize=4, linewidth=1.8)
    if ax_x:
        ax_metric.plot(ax_x, ax_y, label="Val Accuracy", marker="s", markersize=4, linewidth=1.8)
    ax_metric.set_title("Validation Metrics")
    ax_metric.set_xlabel("Optimizer Step")
    ax_metric.set_ylabel("Score")
    ax_metric.set_ylim(0.0, 1.0)
    ax_metric.grid(True, alpha=0.3)
    if fx or ax_x:
        ax_metric.legend()

    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


def plot_confusion_matrix(cm: dict[str, Any] | np.ndarray | list[list[int]], out_path: str | Path) -> Path:
    """Plot annotated 3x3 confusion matrix with raw counts and row-normalized percentages."""
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(cm, dict):
        labels = list(cm["labels"])
        matrix = np.asarray(cm["matrix"], dtype=np.int64)
    else:
        matrix = np.asarray(cm, dtype=np.int64)
        labels = [ID2LABEL[i] for i in range(matrix.shape[0])]

    row_sums = matrix.sum(axis=1, keepdims=True)
    norm = np.divide(
        matrix.astype(np.float64),
        row_sums,
        out=np.zeros_like(matrix, dtype=np.float64),
        where=row_sums > 0,
    )

    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    im = ax.imshow(norm, interpolation="nearest", cmap="Blues", vmin=0.0, vmax=1.0)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ticks = np.arange(len(labels))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title("Confusion Matrix (Counts & Row %)")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            pct = norm[i, j] * 100.0
            color = "white" if norm[i, j] > 0.5 else "black"
            ax.text(
                j,
                i,
                f"{matrix[i, j]:,}\n({pct:.1f}%)",
                ha="center",
                va="center",
                color=color,
                fontsize=9,
            )

    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


def export_error_analysis(
    y_true: np.ndarray | list[int],
    probs: np.ndarray,
    y_pred: np.ndarray | list[int],
    texts_or_indices: list[Any] | np.ndarray,
    out_csv_path: str | Path,
    top_k: int = 50,
) -> Path:
    """Export the top-K highest-confidence misclassifications to a UTF-8 CSV file."""
    y_true_arr = np.asarray(y_true, dtype=np.int64)
    y_pred_arr = np.asarray(y_pred, dtype=np.int64)
    probs_arr = np.asarray(probs, dtype=np.float64)

    if not (len(y_true_arr) == len(y_pred_arr) == len(probs_arr) == len(texts_or_indices)):
        raise ValueError("y_true, probs, y_pred, and texts_or_indices must have equal length")

    mis_idx = np.flatnonzero(y_true_arr != y_pred_arr)
    pred_conf = probs_arr[np.arange(len(y_pred_arr)), y_pred_arr]
    sorted_order = np.argsort(-pred_conf[mis_idx], kind="mergesort")
    top_indices = mis_idx[sorted_order[:top_k]]

    out = Path(out_csv_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "rank",
        "text_or_index",
        "true_label_id",
        "true_label",
        "pred_label_id",
        "pred_label",
        "confidence",
        "prob_negative",
        "prob_neutral",
        "prob_positive",
    ]
    with out.open("w", newline="", encoding="utf-8") as f_csv:
        writer = csv.DictWriter(f_csv, fieldnames=fieldnames)
        writer.writeheader()
        for rank, idx in enumerate(top_indices, start=1):
            t_id = int(y_true_arr[idx])
            p_id = int(y_pred_arr[idx])
            item = texts_or_indices[int(idx)]
            if isinstance(item, np.generic):
                item = item.item()
            writer.writerow(
                {
                    "rank": rank,
                    "text_or_index": item,
                    "true_label_id": t_id,
                    "true_label": ID2LABEL.get(t_id, str(t_id)),
                    "pred_label_id": p_id,
                    "pred_label": ID2LABEL.get(p_id, str(p_id)),
                    "confidence": f"{float(pred_conf[idx]):.6f}",
                    "prob_negative": f"{float(probs_arr[idx, 0]):.6f}",
                    "prob_neutral": f"{float(probs_arr[idx, 1]):.6f}",
                    "prob_positive": f"{float(probs_arr[idx, 2]):.6f}",
                }
            )

    return out


## 3. Smart Dataset Download, Deduplication & Offline CSR Memmap Tokenization

In [7]:
model_cfg = ModelAConfig()
train_cfg = TrainingConfig()

REQUIRED_MEMMAP_FILES = (
    "tokens.bin",
    "offsets.npy",
    "lengths.npy",
    "labels.npy",
    "train_idx.npy",
    "val_idx.npy",
    "test_idx.npy",
    "split_manifest.json",
)


OPTIONAL_MEMMAP_FILES = ("test_texts.jsonl",)


def _has_complete_tokenized_cache(cache_dir: Path) -> bool:
    """Check that all files exist AND that sizes match the manifest (catches truncated copies)."""
    if not all((cache_dir / fname).exists() for fname in REQUIRED_MEMMAP_FILES):
        return False
    try:
        manifest = json.loads((cache_dir / "split_manifest.json").read_text(encoding="utf-8"))
        counts = manifest["counts"]
        if (cache_dir / "tokens.bin").stat().st_size != counts["total_tokens"] * 2:
            return False
        for fname in ("offsets.npy", "lengths.npy", "labels.npy"):
            if len(np.load(cache_dir / fname, mmap_mode="r")) != counts["deduped"]:
                return False
        for split in ("train", "val", "test"):
            if len(np.load(cache_dir / f"{split}_idx.npy", mmap_mode="r")) != manifest["splits"][split]["size"]:
                return False
    except Exception as exc:
        print(f"Cache at {cache_dir} failed integrity check: {exc}")
        return False
    return True


def _copy_cache(src_dir: Path, dst_dir: Path) -> None:
    # Optional files first; split_manifest.json is last in REQUIRED_MEMMAP_FILES so an
    # interrupted copy never looks complete.
    for fname in OPTIONAL_MEMMAP_FILES:
        if (src_dir / fname).exists():
            shutil.copy2(src_dir / fname, dst_dir / fname)
    for fname in REQUIRED_MEMMAP_FILES:
        shutil.copy2(src_dir / fname, dst_dir / fname)


def _configure_kaggle_auth() -> None:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return

    local_kaggle_dir = Path.home() / ".kaggle"
    local_kaggle_json = local_kaggle_dir / "kaggle.json"
    if local_kaggle_json.exists():
        creds = json.loads(local_kaggle_json.read_text(encoding="utf-8"))
        os.environ["KAGGLE_USERNAME"] = creds["username"]
        os.environ["KAGGLE_KEY"] = creds["key"]
        return

    # Fallback 1: Google Colab Secrets.
    try:
        from google.colab import userdata

        k_user = userdata.get("KAGGLE_USERNAME")
        k_key = userdata.get("KAGGLE_KEY")
        if k_user and k_key:
            os.environ["KAGGLE_USERNAME"] = k_user.strip()
            os.environ["KAGGLE_KEY"] = k_key.strip()
            print("Authenticated Kaggle via Colab Secrets.")
            return
    except Exception:
        pass

    # Fallback 2: kaggle.json stored on Google Drive.
    drive_kaggle_json = Path("/content/drive/MyDrive/.kaggle/kaggle.json")
    if drive_kaggle_json.exists():
        local_kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_kaggle_json, local_kaggle_json)
        os.chmod(local_kaggle_json, 0o600)
        creds = json.loads(local_kaggle_json.read_text(encoding="utf-8"))
        os.environ["KAGGLE_USERNAME"] = creds["username"]
        os.environ["KAGGLE_KEY"] = creds["key"]
        print(f"Authenticated Kaggle via {drive_kaggle_json}.")
        return

    # Fallback 3: Interactive browser upload of kaggle.json.
    from google.colab import files

    print("Upload your kaggle.json file to download the Kaggle Yelp dataset:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No kaggle.json uploaded and no Kaggle credentials found.")
    raw_bytes = uploaded.get("kaggle.json") or next(iter(uploaded.values()))
    local_kaggle_dir.mkdir(parents=True, exist_ok=True)
    local_kaggle_json.write_bytes(raw_bytes)
    os.chmod(local_kaggle_json, 0o600)
    creds = json.loads(local_kaggle_json.read_text(encoding="utf-8"))
    os.environ["KAGGLE_USERNAME"] = creds["username"]
    os.environ["KAGGLE_KEY"] = creds["key"]
    print("Authenticated Kaggle via uploaded kaggle.json.")


if _has_complete_tokenized_cache(TOKENIZED_WORK_DIR):
    print(f"Found local tokenized memmap cache in {TOKENIZED_WORK_DIR}; skipping download.")
elif _has_complete_tokenized_cache(TOKENIZED_DRIVE_DIR):
    t0 = time.perf_counter()
    print(f"Copying cached tokenized memmap (~2 GB) from {TOKENIZED_DRIVE_DIR} to {TOKENIZED_WORK_DIR}...")
    _copy_cache(TOKENIZED_DRIVE_DIR, TOKENIZED_WORK_DIR)
    print(f"Restored tokenized memmap from Drive in {time.perf_counter() - t0:.1f}s.")
else:
    _configure_kaggle_auth()
    import kagglehub

    try:
        downloaded = kagglehub.dataset_download(train_cfg.dataset_slug, path=train_cfg.dataset_file)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to download {train_cfg.dataset_slug}/{train_cfg.dataset_file}. "
            f"If you received HTTP 403, visit {train_cfg.dataset_url} and accept the dataset terms first. "
            f"Original error: {exc}"
        ) from exc

    json_path = Path(downloaded)
    if json_path.is_dir():
        json_path = json_path / train_cfg.dataset_file
    assert json_path.exists(), f"Expected downloaded JSON file at {json_path}"
    file_size_bytes = json_path.stat().st_size
    assert file_size_bytes > 5_000_000_000, (
        f"Expected full Yelp review JSON (> 5 GB), got {file_size_bytes / 1e9:.2f} GB at {json_path}"
    )
    print(f"Downloaded {json_path.name} ({file_size_bytes / 1e9:.2f} GB).")

    os.environ["TOKENIZERS_PARALLELISM"] = "true"
    tokenizer = AutoTokenizer.from_pretrained(train_cfg.tokenizer_name)
    t0 = time.perf_counter()
    build_memmap_dataset(
        json_path=json_path,
        out_dir=TOKENIZED_WORK_DIR,
        tokenizer=tokenizer,
        model_cfg=model_cfg,
        train_cfg=train_cfg,
    )
    print(f"Tokenized & indexed full dataset in {time.perf_counter() - t0:.1f}s.")

    # Clean up raw KaggleHub cache on local SSD before backing up memmap files to Drive.
    shutil.rmtree(Path.home() / ".cache" / "kagglehub", ignore_errors=True)

    print(f"Backing up tokenized memmap to Google Drive ({TOKENIZED_DRIVE_DIR})...")
    # Write split_manifest.json last so an interrupted copy cannot look complete on next boot.
    _copy_cache(TOKENIZED_WORK_DIR, TOKENIZED_DRIVE_DIR)
    print("Drive memmap backup complete.")

split_manifest = json.loads((TOKENIZED_WORK_DIR / "split_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(split_manifest["counts"], indent=2))
print(json.dumps(split_manifest["splits"], indent=2))

Using Colab cache for faster access to the 'yelp-dataset' dataset.
Downloaded yelp_academic_dataset_review.json (5.34 GB).


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (514 > 512). Running this sequence through the model will result in indexing errors


Tokenized & indexed full dataset in 838.2s.
Backing up tokenized memmap to Google Drive (/content/drive/MyDrive/yelp_model_a/tokenized)...
Drive memmap backup complete.
{
  "raw": 6990280,
  "dropped_invalid": 0,
  "dropped_duplicate": 16707,
  "deduped": 6973573,
  "total_tokens": 813235540
}
{
  "train": {
    "size": 6913573,
    "class_counts": {
      "negative": 1595427,
      "neutral": 684818,
      "positive": 4633328
    },
    "sha256": "8245a1d054ee7b5adf5d8b1cd4c78e8849269f89555dbdae9ed50909519aa04c"
  },
  "val": {
    "size": 20000,
    "class_counts": {
      "negative": 4615,
      "neutral": 1981,
      "positive": 13404
    },
    "sha256": "54a58e3f134c464edf3fc2f56871d74c3979477d6839d2d0b41f4ff805ea578a"
  },
  "test": {
    "size": 40000,
    "class_counts": {
      "negative": 9231,
      "neutral": 3962,
      "positive": 26807
    },
    "sha256": "7a7188dfbd6112b78b68da5d640b819fe3b98b987acb1f83d5215ca256353241"
  }
}


## 4. Resumable A100 Training (`bf16` + `torch.compile` + O(1) Resume)

In [8]:
# Disable Rust tokenizer thread pool before DataLoader forks worker processes.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(train_cfg.seed)
np.random.seed(train_cfg.seed)
torch.manual_seed(train_cfg.seed)
torch.cuda.manual_seed_all(train_cfg.seed)

train_idx = np.load(TOKENIZED_WORK_DIR / "train_idx.npy")
val_idx = np.load(TOKENIZED_WORK_DIR / "val_idx.npy")
lengths = np.load(TOKENIZED_WORK_DIR / "lengths.npy")

# Precompute exact optimizer step count across all epochs for the warmup + cosine LR schedule.
epoch_batch_counts = [
    len(
        build_epoch_batches(
            train_idx,
            lengths,
            train_cfg.micro_batch_size,
            train_cfg.seed,
            ep,
            mega_batch_mult=train_cfg.mega_batch_mult,
            drop_last=True,
        )
    )
    for ep in range(train_cfg.epochs)
]
total_opt_steps = sum(
    math.ceil(cnt / train_cfg.grad_accum_steps) for cnt in epoch_batch_counts
)
warmup_steps = max(1, int(round(train_cfg.warmup_ratio * total_opt_steps)))
print(f"Total micro-batches per epoch: {epoch_batch_counts} | total_opt_steps={total_opt_steps}, warmup={warmup_steps}")

device = torch.device("cuda")
model = ScratchTransformerClassifier(model_cfg).to(device)
print("Model parameters:", model.count_parameters())

param_groups = build_adamw_param_groups(model, weight_decay=train_cfg.weight_decay)
optimizer = torch.optim.AdamW(
    param_groups,
    lr=train_cfg.peak_lr,
    betas=train_cfg.betas,
    eps=train_cfg.eps,
    fused=True,
)


def _lr_lambda(current_step: int) -> float:
    if current_step < warmup_steps:
        return float(current_step + 1) / float(warmup_steps)
    progress = float(current_step - warmup_steps) / float(max(1, total_opt_steps - warmup_steps))
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return train_cfg.min_lr_ratio + (1.0 - train_cfg.min_lr_ratio) * cosine


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_lambda)
# Label smoothing for training only; validation loss stays plain CE so it remains comparable.
criterion = nn.CrossEntropyLoss(label_smoothing=train_cfg.label_smoothing)
eval_criterion = nn.CrossEntropyLoss()

collate_fn = BucketCollateFn(
    pad_token_id=model_cfg.pad_token_id,
    bucket_multiple=train_cfg.bucket_multiple,
    max_seq_len=model_cfg.max_seq_len,
)

# Use identity-indexed dataset because build_epoch_batches yields global record IDs matching lengths[idx].
full_memmap_ds = YelpMemmapDataset(TOKENIZED_WORK_DIR, np.arange(len(lengths), dtype=np.int64))
val_ds = YelpMemmapDataset(TOKENIZED_WORK_DIR, val_idx)
val_loader = DataLoader(
    val_ds,
    batch_size=train_cfg.micro_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_fn,
)


@torch.inference_mode()
def evaluate_validation(eval_ref_model: nn.Module) -> tuple[float, dict]:
    eval_ref_model.eval()
    total_loss = 0.0
    total_items = 0
    all_preds: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []

    for input_ids, attention_mask, labels in val_loader:
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels_gpu = labels.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = eval_ref_model(input_ids, attention_mask)
            loss = eval_criterion(logits, labels_gpu)
        bsz = int(labels.size(0))
        total_loss += float(loss.item()) * bsz
        total_items += bsz
        all_preds.append(logits.argmax(dim=-1).cpu().numpy())
        all_labels.append(labels.numpy())

    eval_ref_model.train()
    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_labels, axis=0)
    return total_loss / max(total_items, 1), compute_classification_metrics(y_true, y_pred)


def _save_last_checkpoint(
    epoch_val: int,
    next_batch_val: int,
    complete: bool,
) -> None:
    payload = {
        "model_state_dict": strip_orig_mod(model.state_dict()),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch_val,
        "next_batch_idx": next_batch_val,
        "global_step": global_step,
        "best_val_macro_f1": best_val_macro_f1,
        "elapsed_train_sec": elapsed_train_sec,
        "history": history,
        "training_complete": complete,
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all(),
        "rng_numpy": np.random.get_state(),
        "rng_python": random.getstate(),
    }
    atomic_drive_save(payload, WORK_DIR / "last_checkpoint.tmp.pt", last_ckpt_path)


last_ckpt_path = CKPT_DIR / "last_checkpoint.pt"
best_ckpt_path = CKPT_DIR / "best_model.pt"

start_epoch = 0
start_batch_idx = 0
global_step = 0
best_val_macro_f1 = -1.0
elapsed_train_sec = 0.0
history: list[dict] = []
training_complete = False

if last_ckpt_path.exists():
    print(f"Found checkpoint at {last_ckpt_path}; inspecting state...")
    ckpt = torch.load(last_ckpt_path, map_location="cpu", weights_only=False)
    global_step = int(ckpt["global_step"])
    best_val_macro_f1 = float(ckpt["best_val_macro_f1"])
    elapsed_train_sec = float(ckpt["elapsed_train_sec"])
    history = list(ckpt["history"])
    if ckpt.get("training_complete", False):
        training_complete = True
        print(
            f"Training already marked complete (global_step={global_step}, "
            f"best_val_macro_f1={best_val_macro_f1:.4f}, elapsed_train_sec={elapsed_train_sec:.1f}s). Skipping training loop."
        )
    else:
        # Restore state dict onto the uncompiled model before torch.compile so parameter keys match.
        model.load_state_dict(strip_orig_mod(ckpt["model_state_dict"]))
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = int(ckpt["epoch"])
        start_batch_idx = int(ckpt["next_batch_idx"])
        torch.set_rng_state(ckpt["rng_torch"])
        torch.cuda.set_rng_state_all(ckpt["rng_cuda"])
        np.random.set_state(ckpt["rng_numpy"])
        random.setstate(ckpt["rng_python"])
        print(
            f"Resuming from epoch={start_epoch}, next_batch_idx={start_batch_idx}, "
            f"global_step={global_step}, best_val_macro_f1={best_val_macro_f1:.4f}."
        )

if not training_complete:
    torch._dynamo.config.cache_size_limit = 16
    # dynamic=False: compile one static graph per bucket length (64/128/192/256) instead of
    # letting Dynamo fall back to slower dynamic-shape kernels after the 2nd distinct shape.
    compiled_model = torch.compile(model, mode="default", dynamic=False)

    for epoch in range(start_epoch, train_cfg.epochs):
        batches = build_epoch_batches(
            train_idx,
            lengths,
            train_cfg.micro_batch_size,
            train_cfg.seed,
            epoch,
            mega_batch_mult=train_cfg.mega_batch_mult,
            drop_last=True,
        )
        epoch_start_batch = start_batch_idx if epoch == start_epoch else 0
        if epoch_start_batch >= len(batches):
            start_batch_idx = 0
            continue

        batch_sampler = ResumableBatchSampler(batches, start_batch_idx=epoch_start_batch)
        train_loader = DataLoader(
            full_memmap_ds,
            batch_sampler=batch_sampler,
            num_workers=4,
            pin_memory=True,
            persistent_workers=False,
            collate_fn=collate_fn,
        )

        compiled_model.train()
        optimizer.zero_grad(set_to_none=True)
        # Keep loss accumulators on the GPU so we never force a host sync every micro-step.
        accum_loss = torch.zeros((), device=device)
        running_loss = torch.zeros((), device=device)
        running_steps = 0
        seg_start = time.perf_counter()

        pbar = tqdm(
            enumerate(train_loader),
            total=len(batch_sampler),
            desc=f"Epoch {epoch + 1}/{train_cfg.epochs}",
        )
        for step_offset, (input_ids, attention_mask, labels) in pbar:
            micro_batch_idx = epoch_start_batch + step_offset
            # Normalize gradient loss over the actual accumulation window (handles partial tail windows).
            window_start = (micro_batch_idx // train_cfg.grad_accum_steps) * train_cfg.grad_accum_steps
            window_len = min(train_cfg.grad_accum_steps, len(batches) - window_start)

            input_ids = input_ids.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = compiled_model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                scaled_loss = loss / window_len

            scaled_loss.backward()
            accum_loss += loss.detach() / window_len

            is_opt_boundary = (
                ((micro_batch_idx + 1) % train_cfg.grad_accum_steps == 0)
                or (micro_batch_idx + 1 == len(batches))
            )
            if not is_opt_boundary:
                continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            running_loss += accum_loss
            running_steps += 1
            accum_loss.zero_()

            if global_step % 100 == 0:
                pbar.set_postfix(
                    step=global_step,
                    loss=f"{running_loss.item() / max(running_steps, 1):.4f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}",
                )

            if global_step % train_cfg.val_every_steps == 0:
                torch.cuda.synchronize()
                elapsed_train_sec += time.perf_counter() - seg_start
                avg_train_loss = running_loss.item() / max(running_steps, 1)
                running_loss.zero_()
                running_steps = 0

                # Run validation on the uncompiled model (shared weights) to avoid extra Dynamo compilations.
                val_loss, val_metrics = evaluate_validation(model)
                row = {
                    "step": global_step,
                    "epoch": round(epoch + (micro_batch_idx + 1) / len(batches), 4),
                    "train_loss": avg_train_loss,
                    "val_loss": val_loss,
                    "val_macro_f1": val_metrics["macro_f1"],
                    "val_accuracy": val_metrics["accuracy"],
                    "lr": scheduler.get_last_lr()[0],
                }
                history.append(row)
                print(
                    f"[Step {global_step}] train_loss={avg_train_loss:.4f} | "
                    f"val_loss={val_loss:.4f} | val_macro_f1={val_metrics['macro_f1']:.4f} | "
                    f"val_acc={val_metrics['accuracy']:.4f}"
                )
                if val_metrics["macro_f1"] > best_val_macro_f1:
                    best_val_macro_f1 = val_metrics["macro_f1"]
                    atomic_drive_save(
                        {
                            "model_state_dict": strip_orig_mod(model.state_dict()),
                            "model_cfg": asdict(model_cfg),
                            "global_step": global_step,
                            "best_val_macro_f1": best_val_macro_f1,
                        },
                        WORK_DIR / "best_model.tmp.pt",
                        best_ckpt_path,
                    )
                compiled_model.train()
                seg_start = time.perf_counter()

            # Save mid-epoch resume checkpoint on optimizer-step boundaries (end-of-epoch saves separately).
            if (micro_batch_idx + 1 < len(batches)) and (global_step % train_cfg.checkpoint_every_steps == 0):
                torch.cuda.synchronize()
                elapsed_train_sec += time.perf_counter() - seg_start
                _save_last_checkpoint(epoch, micro_batch_idx + 1, complete=False)
                seg_start = time.perf_counter()

        torch.cuda.synchronize()
        elapsed_train_sec += time.perf_counter() - seg_start

        avg_train_loss = (running_loss.item() / running_steps) if running_steps > 0 else None
        val_loss, val_metrics = evaluate_validation(model)
        history.append(
            {
                "step": global_step,
                "epoch": float(epoch + 1),
                "train_loss": avg_train_loss,
                "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "lr": scheduler.get_last_lr()[0],
            }
        )
        print(
            f"[Epoch {epoch + 1} End | Step {global_step}] val_loss={val_loss:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | val_acc={val_metrics['accuracy']:.4f}"
        )
        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            atomic_drive_save(
                {
                    "model_state_dict": strip_orig_mod(model.state_dict()),
                    "model_cfg": asdict(model_cfg),
                    "global_step": global_step,
                    "best_val_macro_f1": best_val_macro_f1,
                },
                WORK_DIR / "best_model.tmp.pt",
                best_ckpt_path,
            )

        is_final_epoch = (epoch + 1 == train_cfg.epochs)
        _save_last_checkpoint(epoch + 1, 0, complete=is_final_epoch)
        start_batch_idx = 0

    training_complete = True
    print(f"Training finished in {elapsed_train_sec / 60:.2f} active min | best_val_macro_f1={best_val_macro_f1:.4f}")

Total micro-batches per epoch: [27006, 27006, 27006] | total_opt_steps=40509, warmup=1620
Model parameters: {'total': 40848899, 'trainable': 40848899, 'embedding': 15627264, 'non_embedding': 25221635}


Epoch 1/3:   0%|          | 0/27006 [00:00<?, ?it/s]

[Step 2000] train_loss=0.5649 | val_loss=0.3471 | val_macro_f1=0.7154 | val_acc=0.8749
[Step 4000] train_loss=0.5038 | val_loss=0.3409 | val_macro_f1=0.7429 | val_acc=0.8802
[Step 6000] train_loss=0.4965 | val_loss=0.3298 | val_macro_f1=0.7517 | val_acc=0.8838
[Step 8000] train_loss=0.4941 | val_loss=0.3242 | val_macro_f1=0.7461 | val_acc=0.8843
[Step 10000] train_loss=0.4917 | val_loss=0.3319 | val_macro_f1=0.7371 | val_acc=0.8864
[Step 12000] train_loss=0.4888 | val_loss=0.3222 | val_macro_f1=0.7590 | val_acc=0.8875
[Epoch 1 End | Step 13503] val_loss=0.3231 | val_macro_f1=0.7450 | val_acc=0.8835


Epoch 2/3:   0%|          | 0/27006 [00:00<?, ?it/s]

[Step 14000] train_loss=0.4876 | val_loss=0.3298 | val_macro_f1=0.7288 | val_acc=0.8827
[Step 16000] train_loss=0.4885 | val_loss=0.3213 | val_macro_f1=0.7548 | val_acc=0.8862
[Step 18000] train_loss=0.4852 | val_loss=0.3164 | val_macro_f1=0.7499 | val_acc=0.8875
[Step 20000] train_loss=0.4846 | val_loss=0.3169 | val_macro_f1=0.7553 | val_acc=0.8883
[Step 22000] train_loss=0.4826 | val_loss=0.3153 | val_macro_f1=0.7538 | val_acc=0.8894
[Step 24000] train_loss=0.4805 | val_loss=0.3107 | val_macro_f1=0.7570 | val_acc=0.8903
[Step 26000] train_loss=0.4787 | val_loss=0.3085 | val_macro_f1=0.7594 | val_acc=0.8914
[Epoch 2 End | Step 27006] val_loss=0.3088 | val_macro_f1=0.7627 | val_acc=0.8914


Epoch 3/3:   0%|          | 0/27006 [00:00<?, ?it/s]

[Step 28000] train_loss=0.4726 | val_loss=0.3061 | val_macro_f1=0.7627 | val_acc=0.8911
[Step 30000] train_loss=0.4740 | val_loss=0.3047 | val_macro_f1=0.7601 | val_acc=0.8930
[Step 32000] train_loss=0.4720 | val_loss=0.3028 | val_macro_f1=0.7546 | val_acc=0.8932
[Step 34000] train_loss=0.4709 | val_loss=0.3026 | val_macro_f1=0.7672 | val_acc=0.8941
[Step 36000] train_loss=0.4710 | val_loss=0.3027 | val_macro_f1=0.7634 | val_acc=0.8941
[Step 38000] train_loss=0.4697 | val_loss=0.3031 | val_macro_f1=0.7668 | val_acc=0.8951
[Step 40000] train_loss=0.4693 | val_loss=0.3010 | val_macro_f1=0.7665 | val_acc=0.8952
[Epoch 3 End | Step 40509] val_loss=0.3005 | val_macro_f1=0.7697 | val_acc=0.8954
Training finished in 81.85 active min | best_val_macro_f1=0.7697


## 5. Post-Hoc Logit Bias Tuning, Held-Out Test Evaluation & Report Artifacts

In [9]:
best_ckpt = torch.load(CKPT_DIR / "best_model.pt", map_location="cpu", weights_only=False)
eval_model = ScratchTransformerClassifier(model_cfg).to(device)
eval_model.load_state_dict(strip_orig_mod(best_ckpt["model_state_dict"]))
eval_model.eval()

eval_collate_fn = BucketCollateFn(
    pad_token_id=model_cfg.pad_token_id,
    bucket_multiple=train_cfg.bucket_multiple,
    max_seq_len=model_cfg.max_seq_len,
)


@torch.inference_mode()
def collect_logits(indices: np.ndarray, batch_size: int = 256) -> tuple[np.ndarray, np.ndarray]:
    ds = YelpMemmapDataset(TOKENIZED_WORK_DIR, indices)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=eval_collate_fn,
    )
    logits_chunks: list[np.ndarray] = []
    labels_chunks: list[np.ndarray] = []
    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = eval_model(input_ids, attention_mask)
        logits_chunks.append(logits.float().cpu().numpy())
        labels_chunks.append(labels.numpy())
    return np.concatenate(logits_chunks, axis=0), np.concatenate(labels_chunks, axis=0)


# Tune 2D additive logit biases strictly on the validation set.
val_logits, val_labels = collect_logits(val_idx, batch_size=train_cfg.micro_batch_size)
logit_biases, tuned_val_f1 = tune_logit_biases(val_logits, val_labels)
raw_val_metrics = compute_classification_metrics(val_labels, val_logits.argmax(axis=-1))
tuned_val_metrics = compute_classification_metrics(
    val_labels, (val_logits + logit_biases).argmax(axis=-1)
)
print(
    f"Validation Macro-F1: raw={raw_val_metrics['macro_f1']:.4f} -> "
    f"logit-adjusted={tuned_val_metrics['macro_f1']:.4f} | biases={logit_biases.tolist()}"
)

# Evaluate once on the held-out test set.
test_idx = np.load(TOKENIZED_WORK_DIR / "test_idx.npy")
test_logits, test_labels = collect_logits(test_idx, batch_size=train_cfg.micro_batch_size)

raw_test_preds = test_logits.argmax(axis=-1)
adj_test_logits = test_logits + logit_biases
adj_test_preds = adj_test_logits.argmax(axis=-1)
adj_test_probs = torch.softmax(torch.from_numpy(adj_test_logits), dim=-1).numpy()

raw_test_metrics = compute_classification_metrics(test_labels, raw_test_preds)
adj_test_metrics = compute_classification_metrics(test_labels, adj_test_preds)

# Benchmark synchronized GPU inference latency under bf16 autocast for bs=1 and bs=32.
test_ds = YelpMemmapDataset(TOKENIZED_WORK_DIR, test_idx)
sample_batches_bs1 = [eval_collate_fn([test_ds[i]])[:2] for i in range(16)]
sample_batches_bs32 = [
    eval_collate_fn([test_ds[i] for i in range(b * 32, (b + 1) * 32)])[:2]
    for b in range(4)
]
with torch.amp.autocast("cuda", dtype=torch.bfloat16):
    latency_stats = benchmark_gpu_latency(
        eval_model,
        sample_batches_bs1=sample_batches_bs1,
        sample_batches_bs32=sample_batches_bs32,
        device=device,
        warmup=20,
        steps=200,
    )

# Error analysis text: prefer original review text (exported during preprocessing, same order as
# test_idx); fall back to decoding truncated token slices for caches built before that export existed.
mis_positions = np.flatnonzero(adj_test_preds != test_labels)
decoded_texts = [""] * len(test_labels)
test_texts_path = TOKENIZED_WORK_DIR / "test_texts.jsonl"
if test_texts_path.exists():
    with test_texts_path.open("r", encoding="utf-8") as f_txt:
        raw_test_texts = [json.loads(line)["text"] for line in f_txt]
    assert len(raw_test_texts) == len(test_labels), "test_texts.jsonl is out of sync with test_idx"
    for pos in mis_positions:
        decoded_texts[int(pos)] = raw_test_texts[int(pos)]
    error_text_source = "original_review_text"
else:
    tokenizer = AutoTokenizer.from_pretrained(train_cfg.tokenizer_name)
    for pos in mis_positions:
        tok_ids, _ = test_ds[int(pos)]
        decoded_texts[int(pos)] = tokenizer.decode(tok_ids.tolist(), skip_special_tokens=True)
    error_text_source = "decoded_truncated_tokens (rebuild cache to get original text)"
print(f"Error analysis text source: {error_text_source}")

# Persist all report artifacts to Google Drive.
config_payload = {
    "model_config": asdict(model_cfg),
    "training_config": asdict(train_cfg),
}
(ARTIFACTS_DIR / "config.json").write_text(json.dumps(config_payload, indent=2), encoding="utf-8")

split_manifest = json.loads((TOKENIZED_WORK_DIR / "split_manifest.json").read_text(encoding="utf-8"))
metrics_payload = {
    "dataset": {
        "slug": split_manifest["dataset_slug"],
        "file": split_manifest["dataset_file"],
        "url": split_manifest["dataset_url"],
        "license": split_manifest["dataset_license"],
        "citation": split_manifest["citation"],
    },
    "split_manifest": split_manifest,
    "parameters": eval_model.count_parameters(),
    "training": {
        "epochs": train_cfg.epochs,
        "global_step": global_step,
        "elapsed_train_sec": elapsed_train_sec,
        "best_val_macro_f1": best_val_macro_f1,
    },
    "hardware": {
        "gpu_name": torch.cuda.get_device_name(0),
        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2),
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "bf16": True,
        "tf32": True,
    },
    "logit_bias_tuning": {
        "biases": logit_biases.tolist(),
        "val_raw": raw_val_metrics,
        "val_logit_adjusted": tuned_val_metrics,
    },
    "test_raw": raw_test_metrics,
    "test_logit_adjusted": adj_test_metrics,
    "gpu_latency_bf16": latency_stats,
    "notes": {
        "val_reuse": (
            "The validation set was used both for best-checkpoint selection (Macro-F1) and for "
            "logit-bias tuning, so 'val_logit_adjusted' is optimistic. The held-out test set was "
            "used exactly once and is the number to report."
        ),
        "latency_padding": (
            f"Inputs are padded to multiples of {train_cfg.bucket_multiple} tokens (min "
            f"{train_cfg.bucket_multiple}), so bs=1 latency reflects >= {train_cfg.bucket_multiple}-token inputs."
        ),
        "error_analysis_text_source": error_text_source,
    },
}
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

history_fields = ["step", "epoch", "train_loss", "val_loss", "val_macro_f1", "val_accuracy", "lr"]
with (ARTIFACTS_DIR / "training_history.csv").open("w", newline="", encoding="utf-8") as f_hist:
    writer = csv.DictWriter(f_hist, fieldnames=history_fields)
    writer.writeheader()
    for row in history:
        writer.writerow({k: row.get(k, "") for k in history_fields})

plot_training_curves(history, ARTIFACTS_DIR / "training_curves.png")
plot_confusion_matrix(adj_test_metrics["confusion_matrix"], ARTIFACTS_DIR / "confusion_matrix.png")
export_error_analysis(
    y_true=test_labels,
    probs=adj_test_probs,
    y_pred=adj_test_preds,
    texts_or_indices=decoded_texts,
    out_csv_path=ARTIFACTS_DIR / "error_analysis_top50.csv",
    top_k=50,
)

print("\n=== Held-Out Test Summary ===")
print(f"Parameters: {eval_model.count_parameters()}")
print(f"Elapsed Active Training Time: {elapsed_train_sec / 60:.2f} min")
print(
    f"Test (Raw Argmax)      -> Acc: {raw_test_metrics['accuracy']:.4f} | "
    f"Macro-F1: {raw_test_metrics['macro_f1']:.4f} | Weighted-F1: {raw_test_metrics['weighted_f1']:.4f}"
)
print(
    f"Test (Logit-Adjusted)  -> Acc: {adj_test_metrics['accuracy']:.4f} | "
    f"Macro-F1: {adj_test_metrics['macro_f1']:.4f} | Weighted-F1: {adj_test_metrics['weighted_f1']:.4f}"
)
print(f"GPU Latency (bs=1):  {latency_stats['bs1']}")
print(f"GPU Latency (bs=32): {latency_stats['bs32']}")
print(f"Artifacts saved to: {ARTIFACTS_DIR}")

Validation Macro-F1: raw=0.7697 -> logit-adjusted=0.7925 | biases=[0.38958333333333284, 0.8968749999999999, 0.0]
Error analysis text source: original_review_text

=== Held-Out Test Summary ===
Parameters: {'total': 40848899, 'trainable': 40848899, 'embedding': 15627264, 'non_embedding': 25221635}
Elapsed Active Training Time: 81.85 min
Test (Raw Argmax)      -> Acc: 0.8959 | Macro-F1: 0.7714 | Weighted-F1: 0.8886
Test (Logit-Adjusted)  -> Acc: 0.8870 | Macro-F1: 0.7902 | Weighted-F1: 0.8912
GPU Latency (bs=1):  {'mean_ms_per_batch': 8.285149079974872, 'p50_ms_per_batch': 8.249055999840493, 'p95_ms_per_batch': 8.466295450534744, 'mean_ms_per_sample': 8.285149079974872, 'samples_per_sec': 120.69788851681507}
GPU Latency (bs=32): {'mean_ms_per_batch': 10.727150955008256, 'p50_ms_per_batch': 10.713748499711073, 'p95_ms_per_batch': 10.76226445075008, 'mean_ms_per_sample': 0.335223467344008, 'samples_per_sec': 2983.0847103964684}
Artifacts saved to: /content/drive/MyDrive/yelp_model_a/artifa

In [10]:
import copy
import json
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"])

import numpy as np
import onnx
import onnxruntime as ort
import torch
import torch.nn as nn

ONNX_DIR = ARTIFACTS_DIR / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = ONNX_DIR / "model_a.onnx"


class OnnxExportWrapper(nn.Module):
    def __init__(self, base_model: nn.Module, biases: np.ndarray | None = None):
        super().__init__()
        self.model = base_model
        if biases is not None:
            self.register_buffer("logit_biases", torch.tensor(biases, dtype=torch.float32))
        else:
            self.logit_biases = None

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        logits = self.model(input_ids, attention_mask)
        if self.logit_biases is not None:
            logits = logits + self.logit_biases
        return logits


# 1. Export to ONNX (fp32 on CPU, dynamic batch & sequence length)
export_model = OnnxExportWrapper(
    copy.deepcopy(eval_model).cpu().float().eval(),
    biases=logit_biases,
).eval()

dummy_ids = torch.randint(1, model_cfg.vocab_size, (2, 32), dtype=torch.long)
dummy_mask = torch.ones((2, 32), dtype=torch.long)

torch.onnx.export(
    export_model,
    (dummy_ids, dummy_mask),
    str(ONNX_PATH),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "seq_len"},
        "attention_mask": {0: "batch_size", 1: "seq_len"},
        "logits": {0: "batch_size"},
    },
    dynamo=False,
)

# 2. Validate graph & verify ONNX Runtime parity with PyTorch
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

test_ids = torch.randint(1, model_cfg.vocab_size, (3, 64), dtype=torch.long)
test_mask = torch.ones((3, 64), dtype=torch.long)
test_mask[0, 40:] = 0
test_ids[0, 40:] = model_cfg.pad_token_id

with torch.inference_mode():
    pt_logits = export_model(test_ids, test_mask).numpy()

ort_logits = session.run(
    ["logits"],
    {"input_ids": test_ids.numpy(), "attention_mask": test_mask.numpy()},
)[0]

max_diff = float(np.max(np.abs(pt_logits - ort_logits)))
assert max_diff < 1e-4, f"ONNX Runtime mismatch too large: {max_diff}"

# 3. Save tokenizer & inference config for your app
tokenizer.save_pretrained(ONNX_DIR)
(ONNX_DIR / "inference_config.json").write_text(
    json.dumps(
        {
            "max_seq_len": model_cfg.max_seq_len,
            "head_tokens": model_cfg.head_tokens,
            "tail_tokens": model_cfg.tail_tokens,
            "pad_token_id": model_cfg.pad_token_id,
            "id2label": {str(k): v for k, v in ID2LABEL.items()},
            "logit_biases_baked_in": True,
            "logit_biases": logit_biases.tolist(),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Export complete & verified (max_diff={max_diff:.2e}). Saved to: {ONNX_DIR}")

/tmp/ipykernel_537/133673479.py:44: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Export complete & verified (max_diff=4.77e-07). Saved to: /content/drive/MyDrive/yelp_model_a/artifacts_v2/onnx
